# Evaluate the fine-tuned Gemma 4 E2B + RICO/OASST adapter

Standalone eval notebook — does **not** retrain. Loads the adapter you already
saved, runs:

1. **RICO captioning** — corpus BLEU-4 (5 refs) + best-of-5 ROUGE-L on `ds["test"]`.
2. **OASST chat** — ROUGE-L on the OASST `validation` split.
3. **Base vs finetuned** — same eval set, LoRA toggled via `model.disable_adapter()`.

**VRAM**: Gemma-4 E2B at 4-bit needs ~4-5 GB — fits on any modern GPU (4090, 3090, A6000, T4, etc.). Loading flags below match what training used (4-bit QLoRA for E-series).

In [3]:
# Uncomment on a fresh machine.
!pip install --upgrade --no-cache-dir "unsloth>=2026.4" "unsloth_zoo>=2026.4"
!pip install --no-cache-dir "transformers>=4.50.0" "trl>=0.14.0" "datasets>=3.2.0"
!pip install --no-cache-dir "peft>=0.14.0" "accelerate>=1.0.0" "bitsandbytes>=0.45.0"
!pip install --no-cache-dir "evaluate>=0.4.0" "sacrebleu>=2.4.0" "rouge_score>=0.1.2" "nltk>=3.9"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 8.4 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.3.3
    Uninstalling huggingface_hub-1.3.3:
      Successfully uninstalled huggingface_hub-1.3.3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 10.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 9.4 MB/s  0:00:00 eta 0:00:01
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24988 sha256=d3190e3efde5b7768493ce62ec3446b5d32118f69c3e36f73b043d7533339f5b
  Stored in directory: /tmp/pip-ephem-wheel-cache-sumxvaku/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [rouge_score] [nltk]bleu]


## 1. Config & imports


In [4]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HUB_DISABLE_XET"] = "1"   # huggingface_hub 1.13 / hf_xet 1.4 signature mismatch

# Path to the locally-saved E2B adapter.
ADAPTER_PATH = "/media/mesut/2Depo/works/gemma4/gemma4_e2b_rico_adapter"

EVAL_N = 100          # RICO test screens to score
CHAT_EVAL_N = 50      # OASST validation chains to score
MAX_NEW_TOKENS_CAP = 64
MAX_NEW_TOKENS_CHAT = 200

# Must match what was used at training time.
SYSTEM_PROMPT = (
    "You are a mobile UI assistant. You look at app screenshots and describe "
    "what the screen does, what the user can do on it, and answer follow-up "
    "questions. Be specific about UI elements and the screen's purpose."
)
EVAL_PROMPT = "Summarise this app screen in one sentence."

import random
import torch
import evaluate
from PIL import Image
from datasets import load_dataset
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

assert torch.cuda.is_available(), "No GPU detected."
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB")
if vram_gb < 6:
    print("WARNING: <6 GB VRAM. E2B at 4-bit may not fit.")


GPU: NVIDIA GeForce RTX 4090 | VRAM: 25.3 GB


## 2. Load adapter + base model

`FastModel.from_pretrained(ADAPTER_PATH)` reads `adapter_config.json` to find
the base model name (`unsloth/gemma-4-E2B-it`), pulls it from HF if needed,
and applies the LoRA on top.

In [5]:
model, tokenizer = FastModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=2048,
    load_in_4bit=True,         # match E-series training (QLoRA, 4-bit base)
    load_in_16bit=False,
    full_finetuning=False,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
FastModel.for_inference(model)
print(f"[MEM] after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")


==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.524 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 2011/2011 [00:02<00:00, 756.96it/s] 


[MEM] after load: 8.26 GB


## 3. Helper: generate

In [7]:
def generate_caption(image: Image.Image, prompt: str = EVAL_PROMPT) -> str:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ]},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS_CAP,
            do_sample=False, use_cache=True,
        )
    plen = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][plen:], skip_special_tokens=True).strip()


def chat_complete(prefix_msgs):
    inputs = tokenizer.apply_chat_template(
        prefix_msgs, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,
    ).to("cuda")
    with torch.inference_mode():
        out = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS_CHAT,
            do_sample=False, use_cache=True,
        )
    plen = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][plen:], skip_special_tokens=True).strip()

## 4. RICO test set: generate + score (finetuned)

In [8]:
ds = load_dataset("rootsautomation/RICO-Screen2Words")
test_rows = ds["test"].shuffle(seed=3407).select(range(EVAL_N))

predictions, references = [], []
for i, row in enumerate(test_rows):
    img = row["image"].convert("RGB")
    pred = generate_caption(img)
    predictions.append(pred)
    references.append(list(row["captions"]))
    if i < 5 or i % 25 == 0:
        print(f"[{i:>3}] pred: {pred}")
        print(f"      ref0: {row['captions'][0]}")

[  0] pred: This screen shows the settings for a Japanese learning app, allowing users to adjust sound playback, reminders, text size, and language settings.
      ref0: display of settings options for a language learning app
[  1] pred: This screen is the homepage of BuzzFeed, featuring trending articles, a cookbook section, and various quizzes.
      ref0: display of news stories in a online media app
[  2] pred: This screen displays the notification and chat preferences for the app, allowing users to customize settings for notifications and chat behavior.
      ref0: page displaying preferences in the phone
[  3] pred: This screen informs the user that a new version is available and prompts them to update the app for a better experience.
      ref0: page asking to update an app
[  4] pred: This screen is the TV shows section of the TurboTax Tax Return App, which currently shows "No Video Found!" and prompts the user to press the floating plus sign to add videos to the Video Locker.


In [9]:
bleu = evaluate.load("sacrebleu")
rouge = evaluate.load("rouge")

# evaluate/sacrebleu wants references shaped [num_examples][num_refs].
max_refs = max(len(r) for r in references)
padded_refs = [list(r) + [""] * (max_refs - len(r)) for r in references]
ft_bleu = bleu.compute(predictions=predictions, references=padded_refs)["score"]

rouge_per_example = []
for pred, refs in zip(predictions, references):
    best = 0.0
    for r in refs:
        best = max(best, rouge.compute(predictions=[pred], references=[r])["rougeL"])
    rouge_per_example.append(best)
ft_rouge_rico = sum(rouge_per_example) / len(rouge_per_example) * 100

print("=" * 60)
print(f"Finetuned RICO BLEU-4 (corpus, 5 refs): {ft_bleu:.2f}")
print(f"Finetuned RICO ROUGE-L (best-of-5):      {ft_rouge_rico:.2f}")
print("=" * 60)


Finetuned RICO BLEU-4 (corpus, 5 refs): 2.01
Finetuned RICO ROUGE-L (best-of-5):      20.75


## 5. OASST chat eval (finetuned)

In [10]:
oasst_val = load_dataset("OpenAssistant/oasst1", split="validation")
val_msg_by_id = {
    m["message_id"]: {
        "text": m["text"], "role": m["role"],
        "parent_id": m["parent_id"], "rank": m["rank"],
    }
    for m in oasst_val
}


def val_walk(mid):
    chain, cur = [], mid
    while cur is not None and cur in val_msg_by_id:
        chain.append(val_msg_by_id[cur]); cur = val_msg_by_id[cur]["parent_id"]
    chain.reverse(); return chain


val_chains = []
for m in oasst_val:
    if m["role"] != "assistant":
        continue
    if m["rank"] is not None and m["rank"] > 0:
        continue
    chain = val_walk(m["message_id"])
    if len(chain) < 2:
        continue
    if not all(x["role"] == ("prompter" if i % 2 == 0 else "assistant") for i, x in enumerate(chain)):
        continue
    val_chains.append(chain)

random.seed(3407)
random.shuffle(val_chains)
val_chains = val_chains[:CHAT_EVAL_N]
print(f"OASST eval chains: {len(val_chains)}")

chat_preds, chat_golds, chat_prompts = [], [], []
for chain in val_chains:
    gold = chain[-1]["text"]
    prefix = chain[:-1]
    msgs = [
        {
            "role": "user" if x["role"] == "prompter" else "assistant",
            "content": [{"type": "text", "text": x["text"]}],
        }
        for x in prefix
    ]
    chat_preds.append(chat_complete(msgs))
    chat_golds.append(gold)
    chat_prompts.append(prefix[-1]["text"])

ft_chat_rouge = rouge.compute(predictions=chat_preds, references=chat_golds)["rougeL"] * 100
print("=" * 60)
print(f"Finetuned OASST chat ROUGE-L (n={len(chat_preds)}): {ft_chat_rouge:.2f}")
print("=" * 60)

OASST eval chains: 50
Finetuned OASST chat ROUGE-L (n=50): 17.09


## 6. Base vs finetuned (LoRA disabled)

In [11]:
print(f"Generating base predictions on {len(test_rows)} RICO screens...")
base_rico_preds = []
with model.disable_adapter():
    for i, row in enumerate(test_rows):
        img = row["image"].convert("RGB")
        base_rico_preds.append(generate_caption(img))
        if i < 3 or i % 25 == 0:
            print(f"  [{i:>3}] base: {base_rico_preds[-1]}")

    print(f"\nGenerating base predictions on {len(val_chains)} OASST chains...")
    base_chat_preds = []
    for chain in val_chains:
        prefix = chain[:-1]
        msgs = [
            {
                "role": "user" if x["role"] == "prompter" else "assistant",
                "content": [{"type": "text", "text": x["text"]}],
            }
            for x in prefix
        ]
        base_chat_preds.append(chat_complete(msgs))

Generating base predictions on 100 RICO screens...
  [  0] base: This screen is the settings menu for a Japanese learning app, allowing users to customize features like sound playback, reminders, text size, and language settings.
  [  1] base: This screen appears to be the homepage of a mobile app called "BuzzFeed," featuring trending articles, curated content like a cookbook, and various quiz sections.
  [  2] base: This screen displays various settings for common preferences, notification preferences, and chat preferences, allowing the user to configure themes, notifications, and chat behavior.
  [ 25] base: This screen displays a photo gallery or feed, featuring a main image related to a news story, and thumbnails of other articles or photos below.
  [ 50] base: This is a login screen for the VIA application where users can enter their email, plate number, and password to log in or sign up.
  [ 75] base: This screen is a sign-up page titled "CREATE ACCOUNT" where the user can enter 

In [12]:
base_bleu = bleu.compute(predictions=base_rico_preds, references=padded_refs)["score"]
base_rouge_per = []
for pred, refs in zip(base_rico_preds, references):
    best = 0.0
    for r in refs:
        best = max(best, rouge.compute(predictions=[pred], references=[r])["rougeL"])
    base_rouge_per.append(best)
base_rouge_rico = sum(base_rouge_per) / len(base_rouge_per) * 100
base_chat_rouge = rouge.compute(predictions=base_chat_preds, references=chat_golds)["rougeL"] * 100

print("=" * 60)
print(f"{'metric':<22}{'base':>10}{'finetuned':>14}{'delta':>10}")
print("-" * 60)
def _row(label, b, f):
    print(f"{label:<22}{b:>10.2f}{f:>14.2f}{f - b:>+10.2f}")
_row("RICO BLEU-4",        base_bleu,       ft_bleu)
_row("RICO ROUGE-L",       base_rouge_rico, ft_rouge_rico)
_row("OASST chat ROUGE-L", base_chat_rouge, ft_chat_rouge)
print("=" * 60)

print("\nRICO samples (REF / BASE / FT):")
for i in range(min(3, len(base_rico_preds))):
    print(f"\n--- rico {i} ---")
    print("REF :", test_rows[i]["captions"][0])
    print("BASE:", base_rico_preds[i])
    print("FT  :", predictions[i])

print("\nOASST chat samples (USER / BASE / FT / GOLD):")
for i in range(min(3, len(base_chat_preds))):
    print(f"\n--- oasst {i} ---")
    print("USER:", chat_prompts[i][:250])
    print("BASE:", base_chat_preds[i][:350])
    print("FT  :", chat_preds[i][:350])
    print("GOLD:", chat_golds[i][:350])

metric                      base     finetuned     delta
------------------------------------------------------------
RICO BLEU-4                 1.79          2.01     +0.22
RICO ROUGE-L               19.32         20.75     +1.43
OASST chat ROUGE-L         16.95         17.09     +0.14

RICO samples (REF / BASE / FT):

--- rico 0 ---
REF : display of settings options for a language learning app
BASE: This screen is the settings menu for a Japanese learning app, allowing users to customize features like sound playback, reminders, text size, and language settings.
FT  : This screen shows the settings for a Japanese learning app, allowing users to adjust sound playback, reminders, text size, and language settings.

--- rico 1 ---
REF : display of news stories in a online media app
BASE: This screen appears to be the homepage of a mobile app called "BuzzFeed," featuring trending articles, curated content like a cookbook, and various quiz sections.
FT  : This screen is the homepage of Buz

## 7. (optional) Test on your own app screenshots

In [ ]:
# CUSTOM_IMAGE = "/path/to/your_app_screenshot.png"
# img = Image.open(CUSTOM_IMAGE).convert("RGB")
# print(generate_caption(img, "Describe what this screen does."))